In [ ]:
from pathlib import Path

# path setting
BASE_PATH = Path.cwd().parent.parent

RAW = BASE_PATH / 'Data' / 'raw'
TEMP = BASE_PATH / 'Data' / 'temp'
USE = BASE_PATH / 'Data' / 'use'
FIGURES = BASE_PATH / 'Results' / 'Figures'
TABLES = BASE_PATH / 'Results' / 'Tables'

for path in [RAW, TEMP, USE, FIGURES, TABLES]:
    path.mkdir(parents=True, exist_ok=True)

print(f"✅ BASE_PATH: {BASE_PATH}")


In [ ]:
# Figure 1. Global data center distribution and growth trends.
# a, Geographical distribution of data centers by country. 
# b, Annual new installations by country and data center type from 2000 to 2024. 

In [ ]:
# Fig. 1a. Geographical distribution of data centers by country/territory

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 6,
    "axes.linewidth": 0.5,
})

# -----------------------------
# 1. Read and clean data
# -----------------------------

df_centers = pd.read_excel(
    RAW / "SPGlobal_Export.xlsx",
    sheet_name="Sheet1"
)

df_country = pd.read_excel(
    RAW / "SPGlobal_Export.xlsx",
    sheet_name="country"
)

df_centers = df_centers.loc[
    df_centers["YR_BUILT"].notna() &
    df_centers["LATITUDE"].notna() &
    df_centers["LONGITUDE"].notna() &
    (df_centers["YR_BUILT"] != 2025)
].copy()

df_merged = df_centers.merge(
    df_country[["COUNTRY", "Alpha-3 code", "Region"]],
    on="COUNTRY",
    how="left"
)

country_counts = (
    df_merged
    .groupby("Alpha-3 code", dropna=False)
    .size()
    .reset_index(name="count")
)

country_counts = country_counts.loc[
    country_counts["Alpha-3 code"].notna()
].copy()

country_counts["Alpha-3 code"] = (
    country_counts["Alpha-3 code"]
    .astype(str)
    .str.upper()
)

# -----------------------------
# 2. Aggregate Hong Kong, Macao and Taiwan into China
# -----------------------------

china_extra_count = country_counts.loc[
    country_counts["Alpha-3 code"].isin(["HKG", "MAC", "TWN"]),
    "count"
].sum()

if "CHN" in country_counts["Alpha-3 code"].values:
    country_counts.loc[
        country_counts["Alpha-3 code"] == "CHN",
        "count"
    ] += china_extra_count
else:
    country_counts = pd.concat(
        [
            country_counts,
            pd.DataFrame({
                "Alpha-3 code": ["CHN"],
                "count": [china_extra_count]
            })
        ],
        ignore_index=True
    )

country_counts.loc[
    country_counts["Alpha-3 code"].isin(["HKG", "MAC", "TWN"]),
    "count"
] = 0

# -----------------------------
# 3. Read local Natural Earth map units
# -----------------------------

map_units_path = (
    RAW /
    "ne_10m_admin_0_map_units" /
    "ne_10m_admin_0_map_units.shp"
)

world_raw = gpd.read_file(map_units_path).copy()

required_columns = ["ISO_A3", "ADM0_A3", "NAME"]

missing_columns = [
    col for col in required_columns
    if col not in world_raw.columns
]

if missing_columns:
    raise ValueError(
        f"Natural Earth fields missing: {missing_columns}\n"
        f"Available fields: {world_raw.columns.tolist()}"
    )

iso_candidate = world_raw["ISO_A3"].astype("string").str.upper()
adm_candidate = world_raw["ADM0_A3"].astype("string").str.upper()

dependency_mask = pd.Series(False, index=world_raw.index)

if "TYPE" in world_raw.columns:
    dependency_mask |= (
        world_raw["TYPE"]
        .astype(str)
        .str.contains("dependency", case=False, na=False)
    )

if "FCLASS_ISO" in world_raw.columns:
    dependency_mask |= (
        world_raw["FCLASS_ISO"]
        .astype(str)
        .str.contains("dependency", case=False, na=False)
    )

valid_iso = iso_candidate.notna() & (iso_candidate != "-99")

# Use territory-specific ISO_A3 whenever it exists:
# GUF for French Guiana, PYF for French Polynesia, GIB for Gibraltar.
# For non-dependency map units with ISO_A3 = -99, such as the four UK
# constituent map units, use ADM0_A3 = GBR.
fallback_to_adm = (
    ~valid_iso &
    ~dependency_mask &
    adm_candidate.notna() &
    (adm_candidate != "-99")
)

world_raw["map_code"] = iso_candidate.where(
    valid_iso,
    adm_candidate.where(fallback_to_adm)
)

world_raw["map_code_source"] = np.select(
    [valid_iso, fallback_to_adm],
    ["ISO_A3", "ADM0_A3"],
    default="unmatched polygon"
)

# Retain unmatched territories as separate no-data polygons rather than
# assigning them to a parent-country count.
unmatched_polygon = world_raw["map_code"].isna()

world_raw.loc[unmatched_polygon, "map_code"] = [
    f"NE_UNMAPPED_{idx}"
    for idx in world_raw.index[unmatched_polygon]
]

world_raw = world_raw.rename(columns={
    "NAME": "name",
    "ADM0_A3": "parent_adm0_a3"
})

# -----------------------------
# 4. Dissolve map units by matching code
# -----------------------------

keep_cols = [
    col for col in [
        "map_code",
        "map_code_source",
        "parent_adm0_a3",
        "GU_A3",
        "name",
        "NAME_LONG",
        "GEOUNIT",
        "ADMIN",
        "SOVEREIGNT",
        "TYPE",
        "FCLASS_ISO"
    ]
    if col in world_raw.columns
]

world = world_raw[keep_cols + ["geometry"]].copy()

world = world.dissolve(
    by="map_code",
    as_index=False,
    aggfunc="first"
)

# -----------------------------
# 5. Merge data-center counts
# -----------------------------

world = world.merge(
    country_counts,
    left_on="map_code",
    right_on="Alpha-3 code",
    how="left"
)

world["count"] = world["count"].fillna(0)
world["display_count"] = world["count"].copy()

# Show HKG, MAC and TWN with China's aggregated value.
china_count = world.loc[
    world["map_code"] == "CHN",
    "count"
]

if len(china_count) > 0:
    china_value = china_count.iloc[0]

    world.loc[
        world["map_code"].isin(["HKG", "MAC", "TWN"]),
        "count"
    ] = 0

    world.loc[
        world["map_code"].isin(["HKG", "MAC", "TWN"]),
        "display_count"
    ] = china_value

world["category"] = np.where(
    world["display_count"] > 0,
    "data",
    "nodata"
)

# -----------------------------
# 6. Diagnostics
# -----------------------------

missing_from_map = (
    country_counts.loc[
        ~country_counts["Alpha-3 code"].isin(world["map_code"]) &
        (country_counts["count"] > 0),
        ["Alpha-3 code", "count"]
    ]
    .sort_values("count", ascending=False)
)

territory_check = world.loc[
    world["map_code"].isin(["GUF", "PYF", "GIB", "HKG", "MAC", "TWN"]),
    [
        col for col in [
            "map_code",
            "map_code_source",
            "parent_adm0_a3",
            "name",
            "count",
            "display_count"
        ]
        if col in world.columns
    ]
].sort_values("map_code")

# -----------------------------
# 7. Plot
# -----------------------------

colors = ["#F1F5F9", "#C7D7EA", "#7FA6CF", "#2F6FAE", "#08306B"]

custom_cmap = LinearSegmentedColormap.from_list(
    "custom_blue",
    colors
)

fig, ax = plt.subplots(figsize=(14, 7), dpi=300)

world_nodata = world.loc[world["category"] == "nodata"]
world_data = world.loc[world["category"] == "data"]

world_nodata.plot(
    ax=ax,
    color="#E8E8E8",
    edgecolor="white",
    linewidth=0.3
)

vmax = (
    world_data["display_count"].quantile(0.95)
    if len(world_data) > 0
    else 1
)

world_data.plot(
    column="display_count",
    ax=ax,
    legend=False,
    cmap=custom_cmap,
    edgecolor="white",
    linewidth=0.3,
    vmin=0,
    vmax=vmax
)

ax.set_xlim([-180, 180])
ax.set_ylim([-60, 85])
ax.axis("off")

sm = plt.cm.ScalarMappable(
    cmap=custom_cmap,
    norm=plt.Normalize(vmin=0, vmax=vmax)
)

sm._A = []

cbar = plt.colorbar(
    sm,
    ax=ax,
    orientation="horizontal",
    fraction=0.03,
    pad=0.04,
    aspect=50
)

cbar.set_label(
    "Number of Data Centers",
    fontsize=18,
    labelpad=8
)

cbar.ax.tick_params(
    labelsize=15,
    width=0.5,
    length=3
)

cbar.outline.set_linewidth(0.5)

ax.legend(
    handles=[
        Patch(
            facecolor="#E8E8E8",
            edgecolor="white",
            label="No Data"
        )
    ],
    loc="lower left",
    fontsize=15,
    frameon=False
)

fig_path = FIGURES / "fig1a_datacenter_heatmap.pdf"

plt.savefig(
    fig_path,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

# -----------------------------
# 8. Source data
# -----------------------------

source_data_fig1a = (
    world
    .drop(columns="geometry")
    .rename(columns={
        "map_code": "Map_unit_code",
        "map_code_source": "Map_code_source",
        "parent_adm0_a3": "Parent_ADM0_A3",
        "name": "Country_or_territory",
        "count": "Data_center_count",
        "display_count": "Map_display_count",
        "category": "Map_category"
    })
    .sort_values(
        ["Map_category", "Map_display_count"],
        ascending=[True, False]
    )
)

source_path = FIGURES / "source_data_fig1a.csv"

source_data_fig1a.to_csv(
    source_path,
    index=False
)

# -----------------------------
# 9. Print checks
# -----------------------------

print(f"\n{'=' * 60}")
print("Data Filtering Summary")
print(f"{'=' * 60}")
print(f"Filtered data centers: {len(df_centers)}")
print(f"Assigned to an Alpha-3 code: {country_counts['count'].sum()}")
print(f"Represented by map units: {world['count'].sum()}")

print("\nTop 10 countries/territories after HKG/MAC/TWN aggregation:")
print(
    country_counts
    .nlargest(10, "count")[["Alpha-3 code", "count"]]
    .to_string(index=False)
)

print("\nCountry codes missing from the map:")
print(
    missing_from_map.to_string(index=False)
    if not missing_from_map.empty
    else "None"
)

print("\nTerritory checks:")
print(
    territory_check.to_string(index=False)
    if not territory_check.empty
    else "None"
)

print(f"\nSaved: {fig_path}")
print(f"Saved: {source_path}")
print(f"{'=' * 60}\n")

In [ ]:
# -----------------------------
# Source data: Fig. 1a
# -----------------------------
source_data_fig1a = (
    world
    .drop(columns="geometry")
    .loc[
        :,
        [
            col for col in [
                "iso_a3",
                "name",
                "parent_iso_a3",
                "count",
                "display_count",
                "category"
            ]
            if col in world.columns
        ]
    ]
    .rename(
        columns={
            "iso_a3": "ISO_A3",
            "name": "Country_or_territory",
            "parent_iso_a3": "Parent_ISO_A3",
            "count": "Data_center_count",
            "display_count": "Map_display_count",
            "category": "Map_category"
        }
    )
    .sort_values(["Map_category", "Map_display_count"], ascending=[True, False])
)

source_data_fig1a.to_csv(
    FIGURES / "source_data_fig1a.csv",
    index=False
)

print(f"Saved: {FIGURES / 'source_data_fig1a.csv'}")

In [ ]:
# -----------------------------
# Mandatory audit: Fig. 1a map and Source Data
# -----------------------------

expected_total = int(country_counts["count"].sum())
mapped_total = int(world["count"].sum())
source_total = int(source_data_fig1a["Data_center_count"].sum())

if not missing_from_map.empty:
    raise AssertionError(
        "Data-center country codes missing from map units:\n"
        f"{missing_from_map.to_string(index=False)}"
    )

if expected_total != mapped_total:
    raise AssertionError(
        f"Map total mismatch: expected {expected_total}, "
        f"mapped {mapped_total}."
    )

if expected_total != source_total:
    raise AssertionError(
        f"Source Data total mismatch: expected {expected_total}, "
        f"source file contains {source_total}."
    )

if not world["map_code"].is_unique:
    raise AssertionError(
        "Map-unit codes are not unique after dissolve."
    )

required_units = ["GUF", "PYF", "GIB", "CHN", "HKG", "MAC", "TWN"]

missing_units = [
    code for code in required_units
    if code not in world["map_code"].values
]

if missing_units:
    raise AssertionError(
        f"Required territory map units missing: {missing_units}"
    )

expected_by_code = (
    country_counts
    .set_index("Alpha-3 code")["count"]
    .to_dict()
)

territory_audit = world.loc[
    world["map_code"].isin(required_units),
    [
        "map_code",
        "name",
        "parent_adm0_a3",
        "map_code_source",
        "count",
        "display_count"
    ]
].copy()

territory_audit["expected_count"] = territory_audit["map_code"].map(
    expected_by_code
).fillna(0)

non_china_territories = ["GUF", "PYF", "GIB"]

if not (
    territory_audit.loc[
        territory_audit["map_code"].isin(non_china_territories),
        "count"
    ]
    .eq(
        territory_audit.loc[
            territory_audit["map_code"].isin(non_china_territories),
            "expected_count"
        ]
    )
    .all()
):
    raise AssertionError(
        "A non-China territory received an incorrect data-center count."
    )

china_display_value = world.loc[
    world["map_code"] == "CHN",
    "count"
].iloc[0]

china_units = territory_audit["map_code"].isin(["HKG", "MAC", "TWN"])

if not territory_audit.loc[china_units, "count"].eq(0).all():
    raise AssertionError(
        "HKG, MAC or TWN retained a separate raw count after aggregation."
    )

if not territory_audit.loc[
    china_units,
    "display_count"
].eq(china_display_value).all():
    raise AssertionError(
        "HKG, MAC or TWN does not display China's aggregated count."
    )

french_guiana = territory_audit.loc[
    territory_audit["map_code"] == "GUF"
].iloc[0]

if french_guiana["parent_adm0_a3"] != "FRA":
    raise AssertionError(
        "French Guiana parent-country code is not FRA."
    )

if french_guiana["count"] != 0:
    raise AssertionError(
        "French Guiana incorrectly received a data-center count."
    )

print("Fig. 1a audit passed.")
print(f"Total data centers represented: {mapped_total}")
print("\nTerritory audit:")
print(
    territory_audit.sort_values("map_code").to_string(index=False)
)

print(
    "\nNote: Map_display_count must not be summed because "
    "China's value is intentionally repeated for HKG, MAC and TWN."
)

In [ ]:
# Fig. 1b. Annual new installations by country and data center type from 2000 to 2024.

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import matplotlib as mpl

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'Arial'
# Nature style settings
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 6,
    'axes.linewidth': 0.5,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'axes.labelsize': 6,
    'legend.fontsize': 6,
})

# Read data
df_centers = pd.read_excel(os.path.join(RAW, 'SPGlobal_Export.xlsx'), sheet_name='Sheet1')
df_country = pd.read_excel(os.path.join(RAW, 'SPGlobal_Export.xlsx'), sheet_name='country')

# Filter valid samples: non-missing YR_BUILT, LATITUDE, LONGITUDE, and YR_BUILT != 2025
df_centers = df_centers[
    (df_centers['YR_BUILT'].notna()) &
    (df_centers['LATITUDE'].notna()) &
    (df_centers['LONGITUDE'].notna()) &
    (df_centers['YR_BUILT'] != 2025)
].copy()

# Merge country information
df_merged = df_centers.merge(
    df_country[['COUNTRY', 'Alpha-3 code', 'Region']],
    on='COUNTRY',
    how='left'
)

# Filter valid years and create year groups
df_time = df_merged[df_merged['YR_BUILT'].notna()].copy()
df_time['Year_Group'] = df_time['YR_BUILT'].apply(
    lambda x: 'Pre-2000' if x < 2000 else str(int(x))
)

# Data center type colors
type_colors = {
    'Hyperscale & Cloud': '#1F3A73',
    'Crypto Mining Data Center': '#3E5F9F',
    'Wholesale Data Center': '#6384C3',
    'Retail Data Center': '#A786C8',
    'Others': '#C1779E'
}

line_color = '#111111'

type_order = [
    'Hyperscale & Cloud',
    'Crypto Mining Data Center',
    'Wholesale Data Center',
    'Retail Data Center',
    'Others'
]

# Categorize types
def categorize_type(type_str):
    if pd.isna(type_str):
        return 'Others'
    type_str = str(type_str).strip()

    if type_str in ['Hyperscale Data Center', 'Cloud Data Center']:
        return 'Hyperscale & Cloud'
    elif type_str in ['Crypto Mining Data Center', 'Wholesale Data Center', 'Retail Data Center']:
        return type_str
    else:
        return 'Others'

df_time['Type_Category'] = df_time['SECONDARY_PPTY_TYPE'].apply(categorize_type)

# Larger data centers for secondary-axis share.
# Crypto is kept as a separate type, not grouped into larger facilities.
larger_categories = ['Hyperscale & Cloud', 'Wholesale Data Center', 'Crypto Mining Data Center']
df_time['Larger_DC'] = df_time['Type_Category'].isin(larger_categories).astype(int)

# ===== Part 1: Stacked bar chart by type + larger share line =====
yearly_type_counts = df_time.groupby(['Year_Group', 'Type_Category']).size().reset_index(name='count')
pivot_yearly = yearly_type_counts.pivot_table(
    index='Year_Group',
    columns='Type_Category',
    values='count',
    fill_value=0
)

# Ensure all types exist
for type_name in type_order:
    if type_name not in pivot_yearly.columns:
        pivot_yearly[type_name] = 0

pivot_yearly = pivot_yearly[type_order]

# Sort years: Pre-2000 first, then 2000-2024
year_labels = ['Pre-2000'] + [str(y) for y in range(2000, 2025)]
pivot_yearly = pivot_yearly.reindex(year_labels, fill_value=0)

# Larger share by year group
yearly_total = pivot_yearly.sum(axis=1)
yearly_larger = pivot_yearly[larger_categories].sum(axis=1)
larger_share = (yearly_larger / yearly_total.replace(0, np.nan)) * 100
larger_share = larger_share.reindex(year_labels)

fig1, ax1 = plt.subplots(figsize=(4.3, 2.7), dpi=300)

x = np.arange(len(pivot_yearly)) * 0.8
width = 0.5
bottom = np.zeros(len(pivot_yearly))

bar_handles = []

for type_name in type_order:
    values = pivot_yearly[type_name].values
    label = type_name.replace(' Data Center', '')
    bars = ax1.bar(
        x, values, width, bottom=bottom,
        label=label,
        color=type_colors[type_name],
        edgecolor='white', linewidth=0.3
    )
    bar_handles.append(bars[0])
    bottom += values

ax1.set_ylabel('Number of New Data Centers')
ax1.set_xlabel('')

# Secondary axis: share of larger data centers
ax2 = ax1.twinx()

line_color = '#F5A000'
line = ax2.plot(
    x,
    larger_share.values,
    color=line_color,
    linewidth=1.15,
    marker='o',
    markersize=3.8,
    markerfacecolor='white',
    markeredgecolor=line_color,
    markeredgewidth=1.0,
    zorder=5,
    label='Larger share'
)

ax2.set_ylabel('Larger data centers (%)', color='black')
ax2.tick_params(axis='y', width=0.5, length=3, colors='black')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_linewidth(0.5)

# Keep right axis clean and comparable
ax2.set_ylim(0, 100)
ax2.set_yticks([0, 25, 50, 75, 100])
ax2.set_yticklabels(['0', '25', '50', '75', '100'])

# X-axis: show Pre-2000 + every other year from 2000
tick_positions = [0] + list(range(1, len(pivot_yearly), 2))
tick_labels = [pivot_yearly.index[i] for i in tick_positions]
ax1.set_xticks([x[i] for i in tick_positions])
ax1.set_xticklabels(tick_labels, rotation=45, ha='right')

ax1.tick_params(axis='both', width=0.5, length=3)

# Combined legend
bar_labels = [t.replace(' Data Center', '') for t in type_order]
handles = bar_handles + [line[0]]
labels = bar_labels + ['Larger share']

ax1.legend(
    handles,
    labels,
    loc='upper left',
    frameon=False,
    bbox_to_anchor=(0.02, 0.98),
    handlelength=1.0,
    handletextpad=0.45,
    labelspacing=0.25,
    borderaxespad=0.0,
    ncol=2
)

ax1.yaxis.grid(True, linestyle='--', alpha=0.25, linewidth=0.45)
ax1.set_axisbelow(True)

ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.spines['left'].set_linewidth(0.5)
ax1.spines['bottom'].set_linewidth(0.5)

# ---------- Era brackets below x-axis ----------
trans = ax1.get_xaxis_transform()

def add_era_bracket(ax, x0, x1, y, label, text_offset=-0.06, lw=0.6):
    ax.plot([x0, x1], [y, y], transform=trans, color='black', lw=lw, clip_on=False)
    ax.plot([x0, x0], [y, y + 0.025], transform=trans, color='black', lw=lw, clip_on=False)
    ax.plot([x1, x1], [y, y + 0.025], transform=trans, color='black', lw=lw, clip_on=False)
    ax.text(
        (x0 + x1) / 2,
        y + text_offset,
        label,
        ha='center',
        va='top',
        transform=trans,
        fontsize=5.5
    )

left_edge = lambda i: x[i] - width / 2
right_edge = lambda i: x[i] + width / 2

add_era_bracket(ax1, left_edge(0), right_edge(6), -0.18, 'Pre-Cloud')
add_era_bracket(ax1, left_edge(7), right_edge(16), -0.18, 'Cloud')
add_era_bracket(ax1, left_edge(17), right_edge(20), -0.18, 'Large-scale ML')
add_era_bracket(ax1, left_edge(21), right_edge(25), -0.18, 'Generative AI')

plt.tight_layout()
plt.subplots_adjust(bottom=0.30)

plt.savefig(
    os.path.join(FIGURES, 'fig1b_datacenter_yearly_stacked_era_clean.tif'),
    dpi=300, bbox_inches='tight', transparent=True,
    pil_kwargs={'compression': 'tiff_lzw'}
)
plt.savefig(
    os.path.join(FIGURES, 'fig1b_datacenter_yearly_stacked_era_clean.pdf'),
    bbox_inches='tight', transparent=True
)

# ===== Part 2: Donut charts (Top 10 countries by count) =====
# Merge Hong Kong, Taiwan, and Macao into China
df_time_merged = df_time.copy()
df_time_merged.loc[df_time_merged['Alpha-3 code'].isin(['HKG', 'TWN', 'MAC']), 'Alpha-3 code'] = 'CHN'

# Automatically get top 10 countries
top10_codes = (
    df_time_merged['Alpha-3 code']
    .value_counts()
    .head(10)
    .index
    .tolist()
)

fig2, axes = plt.subplots(2, 5, figsize=(3, 1.2), dpi=300)
axes = axes.flatten()

for idx, code in enumerate(top10_codes):
    ax = axes[idx]
    country_data = df_time_merged[df_time_merged['Alpha-3 code'] == code]
    type_counts = country_data['Type_Category'].value_counts().reindex(type_order, fill_value=0)
    total = type_counts.sum()

    if total > 0:
        ax.pie(
            type_counts,
            colors=[type_colors[t] for t in type_order],
            startangle=90,
            counterclock=False,
            wedgeprops=dict(width=0.45, edgecolor='white', linewidth=0.8)
        )
    else:
        ax.pie(
            [1],
            colors=['#EEEEEE'],
            wedgeprops=dict(width=0.35, edgecolor='white', linewidth=0.4)
        )

    ax.text(0, 0.05, code, ha='center', va='center', color='black')
    ax.text(0, -0.29, f'{int(total)}', ha='center', va='center', color='gray')
    ax.axis('equal')

plt.tight_layout()

plt.savefig(
    os.path.join(FIGURES, 'fig1b_datacenter_type_donuts.tif'),
    dpi=300, bbox_inches='tight', transparent=True,
    pil_kwargs={'compression': 'tiff_lzw'}
)
plt.savefig(
    os.path.join(FIGURES, 'fig1b_datacenter_type_donuts.pdf'),
    bbox_inches='tight', transparent=True
)

plt.show()

# Print filtering summary
print(f"\n{'='*60}")
print("Data Filtering Summary for Fig. 1b")
print(f"{'='*60}")
print(f"Total data centers after filtering: {len(df_centers)}")
print(f"Data centers with valid year: {len(df_time)}")
print(f"Year range: {df_time['YR_BUILT'].min():.0f} - {df_time['YR_BUILT'].max():.0f}")

print("\nTop 10 countries (after merging HKG/TWN/MAC into CHN):")
print(df_time_merged['Alpha-3 code'].value_counts().head(10))

print("\nData center types distribution:")
print(df_time['Type_Category'].value_counts())

print("\nAnnual larger data-center share (%):")
print(larger_share.round(2))

print(f"{'='*60}\n")

In [ ]:
# -----------------------------
# Source data: Fig. 1b
# -----------------------------

# Panel: annual stacked counts and larger-data-center share
source_data_fig1b_annual = (
    pivot_yearly
    .assign(
        Total_new_data_centers=yearly_total,
        Larger_data_centers=yearly_larger,
        Larger_data_center_share_pct=larger_share
    )
    .reset_index()
    .rename(columns={"Year_Group": "Commissioning_year"})
)

source_data_fig1b_annual.to_csv(
    FIGURES / "source_data_fig1b_annual.csv",
    index=False
)

# Panel: data-center-type composition in the ten largest markets
donut_index = pd.MultiIndex.from_product(
    [top10_codes, type_order],
    names=["Alpha-3 code", "Data_center_type"]
)

source_data_fig1b_top10 = (
    df_time_merged
    .groupby(["Alpha-3 code", "Type_Category"])
    .size()
    .reindex(donut_index, fill_value=0)
    .reset_index(name="Data_center_count")
)

source_data_fig1b_top10["Country_total_data_centers"] = (
    source_data_fig1b_top10
    .groupby("Alpha-3 code")["Data_center_count"]
    .transform("sum")
)

source_data_fig1b_top10["Data_center_share_pct"] = np.where(
    source_data_fig1b_top10["Country_total_data_centers"] > 0,
    100
    * source_data_fig1b_top10["Data_center_count"]
    / source_data_fig1b_top10["Country_total_data_centers"],
    np.nan
)

source_data_fig1b_top10 = source_data_fig1b_top10.rename(
    columns={"Alpha-3 code": "ISO_A3"}
)

source_data_fig1b_top10.to_csv(
    FIGURES / "source_data_fig1b_top10_type_composition.csv",
    index=False
)

print(f"Saved: {FIGURES / 'source_data_fig1b_annual.csv'}")
print(f"Saved: {FIGURES / 'source_data_fig1b_top10_type_composition.csv'}")

In [ ]:
# Limited-sample evidence on data center physical size over time
# Bubble scatter using PPTY_SIZE_AREA merged by PPTY_KEY.
# PPTY_SIZE_AREA definition: total interior area of the building or buildings, in square meters.
# OLS trend is fitted on all observations with reported property size in log10(size).

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 6,
    'axes.linewidth': 0.5,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'axes.labelsize': 7,
    'legend.fontsize': 6,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

# ---------------------------------------------------------------------
# 1. Read and merge data for plotting only
# ---------------------------------------------------------------------

main_path = os.path.join(RAW, 'SPGlobal_Export.xlsx')
size_path = os.path.join(RAW, 'capitaliqpro_datacenter_20260412access.xlsx')

df_main = pd.read_excel(main_path, sheet_name='Sheet1')
df_size = pd.read_excel(size_path, sheet_name='Sheet1')

main_cols = [
    'PPTY_KEY',
    'PPTY_NAME',
    'COUNTRY',
    'SECONDARY_PPTY_TYPE',
    'YR_BUILT',
    'LATITUDE',
    'LONGITUDE'
]
main_cols = [c for c in main_cols if c in df_main.columns]

df_plot = df_main[main_cols].copy()

if 'PPTY_SIZE_AREA' not in df_size.columns:
    raise ValueError("PPTY_SIZE_AREA is not found in capitaliqpro_datacenter_20260412access.xlsx")

df_size_small = df_size[['PPTY_KEY', 'PPTY_SIZE_AREA']].copy()
df_size_small['PPTY_SIZE_AREA'] = pd.to_numeric(
    df_size_small['PPTY_SIZE_AREA'],
    errors='coerce'
)

df_size_small = (
    df_size_small
    .dropna(subset=['PPTY_KEY'])
    .groupby('PPTY_KEY', as_index=False)['PPTY_SIZE_AREA']
    .max()
)

df_plot = df_plot.merge(
    df_size_small,
    on='PPTY_KEY',
    how='left',
    validate='m:1'
)

# ---------------------------------------------------------------------
# 2. Clean and categorize
# ---------------------------------------------------------------------

df_plot = df_plot[
    (df_plot['YR_BUILT'].notna()) &
    (df_plot['YR_BUILT'] != 2025) &
    (df_plot['PPTY_SIZE_AREA'].notna()) &
    (df_plot['PPTY_SIZE_AREA'] > 0)
].copy()

df_plot['YR_BUILT'] = df_plot['YR_BUILT'].astype(int)

df_plot = df_plot[
    (df_plot['YR_BUILT'] >= 2000) &
    (df_plot['YR_BUILT'] <= 2024)
].copy()

df_plot['property_size_sqm'] = df_plot['PPTY_SIZE_AREA']
df_plot['property_size_1000sqm'] = df_plot['property_size_sqm'] / 1000

def categorize_type(type_str):
    if pd.isna(type_str):
        return 'Others'
    type_str = str(type_str).strip()

    if type_str in ['Hyperscale Data Center', 'Cloud Data Center']:
        return 'Hyperscale & Cloud'
    elif type_str in ['Crypto Mining Data Center', 'Wholesale Data Center', 'Retail Data Center']:
        return type_str
    else:
        return 'Others'

df_plot['Type_Category'] = df_plot['SECONDARY_PPTY_TYPE'].apply(categorize_type)

type_order = [
    'Hyperscale & Cloud',
    'Wholesale Data Center',
    'Retail Data Center',
    'Crypto Mining Data Center',
    'Others'
]

type_labels = {
    'Hyperscale & Cloud': 'Hyperscale & cloud',
    'Wholesale Data Center': 'Wholesale',
    'Retail Data Center': 'Retail',
    'Crypto Mining Data Center': 'Crypto mining',
    'Others': 'Others'
}

type_colors = {
    'Hyperscale & Cloud': '#744577',
    'Wholesale Data Center': '#FE9EC7',
    'Retail Data Center': '#89D4FF',
    'Crypto Mining Data Center': '#F9F6C4',
    'Others': '#C9CED6'
}

# ---------------------------------------------------------------------
# 3. Bubble scaling and annual median
# ---------------------------------------------------------------------

area_plot = df_plot['property_size_1000sqm'].astype(float)

area_for_size = area_plot.clip(
    lower=area_plot.quantile(0.02),
    upper=area_plot.quantile(0.98)
)

df_plot['bubble_size'] = np.interp(
    np.sqrt(area_for_size),
    (np.sqrt(area_for_size).min(), np.sqrt(area_for_size).max()),
    (12, 135)
)

annual_stats = (
    df_plot
    .groupby('YR_BUILT')
    .agg(
        median_size_1000sqm=('property_size_1000sqm', 'median'),
        n=('property_size_1000sqm', 'size')
    )
    .reset_index()
)

annual_stats_plot = annual_stats.copy()

# ---------------------------------------------------------------------
# 4. OLS trend on log10(size), clustered by year
# ---------------------------------------------------------------------

trend_df = df_plot[
    (df_plot['property_size_1000sqm'].notna()) &
    (df_plot['property_size_1000sqm'] > 0) &
    (df_plot['YR_BUILT'].notna())
].copy()

trend_df['year_c'] = trend_df['YR_BUILT'] - trend_df['YR_BUILT'].min()
trend_df['log_size'] = np.log10(trend_df['property_size_1000sqm'])

y = trend_df['log_size'].to_numpy(dtype=float)
x = trend_df['year_c'].to_numpy(dtype=float)
groups = trend_df['YR_BUILT'].to_numpy()

X = np.column_stack([np.ones(len(x)), x])

n, k = X.shape
unique_groups = np.unique(groups)
G = len(unique_groups)

if G <= 1:
    raise ValueError("Cluster-robust standard errors require at least two year clusters.")

XtX_inv = np.linalg.pinv(X.T @ X)
beta = XtX_inv @ X.T @ y
resid = y - X @ beta

meat = np.zeros((k, k))
for g in unique_groups:
    idx = groups == g
    Xg = X[idx, :]
    ug = resid[idx]
    score_g = Xg.T @ ug
    meat += np.outer(score_g, score_g)

small_sample_correction = (G / (G - 1)) * ((n - 1) / (n - k))
vcov = small_sample_correction * XtX_inv @ meat @ XtX_inv
se = np.sqrt(np.diag(vcov))

const = beta[0]
slope = beta[1]

df_cluster = G - 1
t_stat = slope / se[1]
pval = 2 * stats.t.sf(np.abs(t_stat), df=df_cluster)

t_crit = stats.t.ppf(0.975, df=df_cluster)
ci_low = slope - t_crit * se[1]
ci_high = slope + t_crit * se[1]

annual_growth = (10 ** slope - 1) * 100
annual_growth_low = (10 ** ci_low - 1) * 100
annual_growth_high = (10 ** ci_high - 1) * 100

sig = (
    '***' if pval < 0.001 else
    '**' if pval < 0.01 else
    '*' if pval < 0.05 else
    '+' if pval < 0.10 else
    ''
)

trend_label = (
    f"OLS trend: {annual_growth:.1f}%/yr{sig}\n"
    f"95% CI: {annual_growth_low:.1f} to {annual_growth_high:.1f}%"
)

x_trend = np.arange(trend_df['YR_BUILT'].min(), trend_df['YR_BUILT'].max() + 1)
x_trend_c = x_trend - trend_df['YR_BUILT'].min()
y_trend = 10 ** (const + slope * x_trend_c)

# ---------------------------------------------------------------------
# 5. Plot
# ---------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(4.4, 3.05), dpi=300)

for t in type_order:
    dat = df_plot[df_plot['Type_Category'] == t]
    if dat.empty:
        continue

    ax.scatter(
        dat['YR_BUILT'],
        dat['property_size_1000sqm'],
        s=dat['bubble_size'],
        c=type_colors[t],
        edgecolors='black',
        linewidths=0.32,
        alpha=0.62,
        label=type_labels[t],
        zorder=3
    )

if len(annual_stats_plot) > 0:
    ax.plot(
        annual_stats_plot['YR_BUILT'],
        annual_stats_plot['median_size_1000sqm'],
        color='black',
        linewidth=1.0,
        marker='o',
        markersize=2.3,
        markerfacecolor='white',
        markeredgecolor='black',
        markeredgewidth=0.6,
        zorder=5,
        label='Annual median'
    )

ax.plot(
    x_trend,
    y_trend,
    color='#D62728',
    linewidth=1.25,
    linestyle='--',
    zorder=6,
    label='OLS trend'
)

ax.set_yscale('log')

ax.set_xlabel('Year built')
ax.set_ylabel('Property size (thousand m²)')

ax.set_xlim(1999.3, 2024.7)
ax.set_xticks([2000, 2005, 2010, 2015, 2020, 2024])
ax.set_xticklabels(['2000', '2005', '2010', '2015', '2020', '2024'])

ax.tick_params(axis='both', width=0.5, length=3)

ax.yaxis.grid(True, linestyle='--', alpha=0.25, linewidth=0.45)
ax.xaxis.grid(False)
ax.set_axisbelow(True)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(0.5)
ax.spines['bottom'].set_linewidth(0.5)

ax.text(
    0.03, 0.05,
    trend_label,
    transform=ax.transAxes,
    ha='left',
    va='bottom',
    fontsize=5.8,
    color='#D62728',
    bbox=dict(
        facecolor='white',
        edgecolor='none',
        alpha=0.72,
        boxstyle='round,pad=0.18'
    )
)

handles, labels = ax.get_legend_handles_labels()

priority_labels = ['Annual median', 'OLS trend']
new_handles, new_labels = [], []

for lab in priority_labels:
    if lab in labels:
        idx = labels.index(lab)
        new_handles.append(handles[idx])
        new_labels.append(labels[idx])

for h, lab in zip(handles, labels):
    if lab not in priority_labels:
        new_handles.append(h)
        new_labels.append(lab)

ax.legend(
    new_handles,
    new_labels,
    loc='lower center',
    bbox_to_anchor=(0.5, 1.02),
    frameon=False,
    handletextpad=0.45,
    columnspacing=0.9,
    labelspacing=0.25,
    borderaxespad=0.0,
    markerscale=0.75,
    ncol=3
)

coverage_text = (
    f"Area available for {len(df_plot):,} facilities "
    f"({len(df_plot) / len(df_main):.1%} of raw records)"
)

fig.text(
    0.98, 0.01,
    coverage_text,
    ha='right',
    va='bottom',
    fontsize=5.7,
    color='black'
)

plt.tight_layout()
plt.subplots_adjust(top=0.82, bottom=0.16)

plt.savefig(
    os.path.join(FIGURES, 'fig1b_datacenter_property_size_bubble_limited_sample_log_trend_sig.tif'),
    dpi=300,
    bbox_inches='tight',
    transparent=True,
    pil_kwargs={'compression': 'tiff_lzw'}
)

plt.savefig(
    os.path.join(FIGURES, 'fig1b_datacenter_property_size_bubble_limited_sample_log_trend_sig.pdf'),
    bbox_inches='tight',
    transparent=True
)

plt.show()

# ---------------------------------------------------------------------
# 6. Export diagnostics
# ---------------------------------------------------------------------

coverage_by_type = (
    df_plot
    .groupby('Type_Category')
    .agg(
        n=('property_size_sqm', 'size'),
        median_size_sqm=('property_size_sqm', 'median'),
        mean_size_sqm=('property_size_sqm', 'mean'),
        p25_size_sqm=('property_size_sqm', lambda x: x.quantile(0.25)),
        p75_size_sqm=('property_size_sqm', lambda x: x.quantile(0.75))
    )
    .reset_index()
    .sort_values('n', ascending=False)
)

coverage_by_year = annual_stats.copy()

coverage_by_type.to_csv(
    os.path.join(TABLES, 'fig1b_property_size_limited_sample_by_type.csv'),
    index=False
)

coverage_by_year.to_csv(
    os.path.join(TABLES, 'fig1b_property_size_limited_sample_by_year.csv'),
    index=False
)

print(f"\n{'='*60}")
print("Property-size bubble figure summary")
print(f"{'='*60}")
print(f"Raw records in SPGlobal_Export.xlsx: {len(df_main):,}")
print(f"Records with matched positive PPTY_SIZE_AREA, valid YR_BUILT, 2000-2024: {len(df_plot):,}")
print(f"Share of raw records used in this limited-sample figure: {len(df_plot) / len(df_main):.2%}")
print("PPTY_SIZE_AREA definition: total interior area of the building or buildings")
print("Displayed unit: thousand square meters")
print("\nOLS trend fitted on all observations with reported property size:")
print(f"Slope = {slope:.4f} log10(thousand m²) per year")
print(f"Implied annual growth = {annual_growth:.2f}% per year")
print(f"95% CI = [{annual_growth_low:.2f}%, {annual_growth_high:.2f}%]")
print(f"P-value = {pval:.4g}")
print(f"N = {len(trend_df):,}, years = {trend_df['YR_BUILT'].nunique()}")

print("\nProperty-size coverage by type:")
print(coverage_by_type)

print("\nAnnual median property-size sample size:")
print(coverage_by_year.tail(10))

print(f"{'='*60}\n")

In [ ]:
# Figure 2. Proximity of data centers to power generation infrastructure. 
# a, Distribution of distance between data centers (by establishment phase: <2006, 2006-2015, 2016-2019, 2020-2024) and their nearest power plants (coal, oil & gas, solar & wind) for the top 10 data center markets. 
# b, Energy source composition of generation capacity for these nearest power plants by country and phase.

In [ ]:
# Fig. 2a calculation

import os
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

top10_countries = ["USA", "CHN", "JPN", "GBR", "DEU", "RUS", "FRA", "CAN", "NLD", "AUS"]

oilgas_df = pd.read_excel(
    os.path.join(RAW, "Global-Oil-and-Gas-Plant-Tracker-GOGPT-January-2025.xlsx"),
    sheet_name="all"
)
oilgas_country = pd.read_excel(
    os.path.join(RAW, "Global-Oil-and-Gas-Plant-Tracker-GOGPT-January-2025.xlsx"),
    sheet_name="country"
)

coal_df = pd.read_excel(
    os.path.join(RAW, "Global-Coal-Plant-Tracker-January-2025.xlsx"),
    sheet_name="Units"
)
coal_country = pd.read_excel(
    os.path.join(RAW, "Global-Coal-Plant-Tracker-January-2025.xlsx"),
    sheet_name="country"
)

solar_df = pd.read_excel(
    os.path.join(RAW, "Global-Solar-Power-Tracker-February-2025.xlsx"),
    sheet_name="all"
)
solar_country = pd.read_excel(
    os.path.join(RAW, "Global-Solar-Power-Tracker-February-2025.xlsx"),
    sheet_name="country"
)

wind_df = pd.read_excel(
    os.path.join(RAW, "Global-Wind-Power-Tracker-February-2025.xlsx"),
    sheet_name="Data"
)
wind_country = pd.read_excel(
    os.path.join(RAW, "Global-Wind-Power-Tracker-February-2025.xlsx"),
    sheet_name="country"
)

ai_df = pd.read_excel(os.path.join(RAW, "SPGlobal_Export.xlsx"), sheet_name="Sheet1")
ai_country = pd.read_excel(os.path.join(RAW, "SPGlobal_Export.xlsx"), sheet_name="country")

def process_plant_data(plant_df, country_df, plant_type):
    plant_clean = plant_df[
        plant_df["Status"].isin(["operating", "retired"])
        & plant_df["Latitude"].notna()
        & plant_df["Longitude"].notna()
    ].copy()

    for col in ["Latitude", "Longitude", "Start year", "Retired year", "Capacity (MW)"]:
        plant_clean[col] = pd.to_numeric(plant_clean[col], errors="coerce")

    plant_clean = plant_clean.dropna(subset=["Latitude", "Longitude"]).merge(
        country_df[["Country/Area", "Alpha-3 code"]],
        on="Country/Area",
        how="left"
    )

    plant_clean["plant_type"] = plant_type
    return plant_clean

all_plants = pd.concat(
    [
        process_plant_data(oilgas_df, oilgas_country, "Oil & Gas"),
        process_plant_data(coal_df, coal_country, "Coal"),
        process_plant_data(solar_df, solar_country, "Solar"),
        process_plant_data(wind_df, wind_country, "Wind")
    ],
    ignore_index=True
)

all_plants = all_plants[
    all_plants["Alpha-3 code"].isin(top10_countries)
].copy()

ai_clean = ai_df[
    ai_df["LATITUDE"].notna()
    & ai_df["LONGITUDE"].notna()
].copy()

ai_clean["LATITUDE"] = pd.to_numeric(ai_clean["LATITUDE"], errors="coerce")
ai_clean["LONGITUDE"] = pd.to_numeric(ai_clean["LONGITUDE"], errors="coerce")
ai_clean["YR_BUILT"] = pd.to_numeric(ai_clean["YR_BUILT"], errors="coerce")

ai_clean = ai_clean.dropna(subset=["LATITUDE", "LONGITUDE", "YR_BUILT"]).merge(
    ai_country[["COUNTRY", "Alpha-3 code"]],
    on="COUNTRY",
    how="left"
)

# Do not merge Hong Kong, Macao or Taiwan into China.
ai_clean = ai_clean[
    ai_clean["Alpha-3 code"].isin(top10_countries)
    & (ai_clean["YR_BUILT"] <= 2024)
].copy()

ai_clean["YR_BUILT"] = ai_clean["YR_BUILT"].astype(int)

def to_unit_sphere(latitude, longitude):
    lat = np.radians(np.asarray(latitude, dtype=float))
    lon = np.radians(np.asarray(longitude, dtype=float))

    return np.column_stack(
        [
            np.cos(lat) * np.cos(lon),
            np.cos(lat) * np.sin(lon),
            np.sin(lat)
        ]
    )

def haversine_paired(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(
        np.radians,
        [np.asarray(lat1), np.asarray(lon1), np.asarray(lat2), np.asarray(lon2)]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

results_parts = []

for country in top10_countries:
    country_dcs = ai_clean[ai_clean["Alpha-3 code"] == country].copy()
    country_plants = all_plants[all_plants["Alpha-3 code"] == country].copy()

    if country_dcs.empty or country_plants.empty:
        continue

    for ai_year, dc_year_data in country_dcs.groupby("YR_BUILT", sort=True):
        operating_mask = (
            (country_plants["Start year"] <= ai_year)
            & (
                (country_plants["Status"] == "operating")
                | (
                    (country_plants["Status"] == "retired")
                    & (country_plants["Retired year"] >= ai_year)
                )
            )
        )

        operating_plants = country_plants.loc[operating_mask].copy()

        if operating_plants.empty:
            continue

        plant_xyz = to_unit_sphere(
            operating_plants["Latitude"].to_numpy(),
            operating_plants["Longitude"].to_numpy()
        )

        dc_xyz = to_unit_sphere(
            dc_year_data["LATITUDE"].to_numpy(),
            dc_year_data["LONGITUDE"].to_numpy()
        )

        nearest_indices = cKDTree(plant_xyz).query(dc_xyz, k=1)[1]
        nearest_plants = operating_plants.iloc[nearest_indices]

        nearest_distances = haversine_paired(
            dc_year_data["LATITUDE"].to_numpy(),
            dc_year_data["LONGITUDE"].to_numpy(),
            nearest_plants["Latitude"].to_numpy(),
            nearest_plants["Longitude"].to_numpy()
        )

        results_parts.append(
            pd.DataFrame(
                {
                    "country": country,
                    "ai_year": ai_year,
                    "nearest_distance_km": nearest_distances
                }
            )
        )

results_df = pd.concat(results_parts, ignore_index=True)

def classify_year(year):
    if year < 2006:
        return "Phase I: <2006"
    if year <= 2015:
        return "Phase II: 2006-2015"
    if year <= 2019:
        return "Phase III: 2016-2019"
    return "Phase IV: 2020-2024"

results_df["year_group"] = pd.Categorical(
    results_df["ai_year"].apply(classify_year),
    categories=[
        "Phase I: <2006",
        "Phase II: 2006-2015",
        "Phase III: 2016-2019",
        "Phase IV: 2020-2024"
    ],
    ordered=True
)

output_path = os.path.join(TEMP, "intermediate_data_top10_mainland_china.csv")
results_df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")


In [ ]:
# Fig. 2a plot: country-specific pooled 99th percentile display sample

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Rectangle

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"

plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 8
plt.rcParams["axes.linewidth"] = 0.8

np.random.seed(123)

period_order = [
    "Phase I: <2006",
    "Phase II: 2006-2015",
    "Phase III: 2016-2019",
    "Phase IV: 2020-2024"
]

top10_countries = ["USA", "CHN", "JPN", "GBR", "DEU", "RUS", "FRA", "CAN", "NLD", "AUS"]

period_colors = {
    "Phase I: <2006": "#A3FFD6",
    "Phase II: 2006-2015": "#7BC9FF",
    "Phase III: 2016-2019": "#8576FF",
    "Phase IV: 2020-2024": "#1C1678"
}

def nice_y_limit(value):
    if value <= 10:
        step = 5
    elif value <= 20:
        step = 10
    elif value <= 50:
        step = 10
    elif value <= 100:
        step = 25
    elif value <= 200:
        step = 50
    elif value <= 500:
        step = 100
    elif value <= 1000:
        step = 200
    else:
        step = 500

    return np.ceil(value / step) * step

df_plot = pd.read_csv(
    os.path.join(TEMP, "intermediate_data_top10_mainland_china.csv")
)

df_plot["year_group"] = pd.Categorical(
    df_plot["year_group"],
    categories=period_order,
    ordered=True
)

df_plot = df_plot[
    df_plot["country"].isin(top10_countries)
    & df_plot["nearest_distance_km"].notna()
].copy()

country_limits = (
    df_plot.groupby("country", observed=True)["nearest_distance_km"]
    .quantile(0.99)
    .reset_index(name="Country pooled 99th percentile (km)")
)

country_limits["Panel y-axis upper limit (km)"] = country_limits[
    "Country pooled 99th percentile (km)"
].apply(nice_y_limit)

df_plot = df_plot.merge(country_limits, on="country", how="left")

df_plot = df_plot[
    df_plot["nearest_distance_km"]
    <= df_plot["Country pooled 99th percentile (km)"]
].copy()

df_plot["country"] = pd.Categorical(
    df_plot["country"],
    categories=top10_countries,
    ordered=True
)

fig = plt.figure(figsize=(13, 4.8), dpi=300)

from matplotlib.gridspec import GridSpec

gs = GridSpec(
    2, 5, figure=fig,
    left=0.06, right=0.85, top=0.86, bottom=0.14,
    hspace=0.40, wspace=0.22
)

for idx, country in enumerate(top10_countries):
    row = idx // 5
    col = idx % 5
    ax = fig.add_subplot(gs[row, col])

    country_data = df_plot[df_plot["country"] == country]

    positions = []
    data_list = []
    colors_list = []
    position_mapping = {0: 0, 1: 0.45, 2: 0.9, 3: 1.35}

    for i, period in enumerate(period_order):
        period_data = country_data.loc[
            country_data["year_group"] == period,
            "nearest_distance_km"
        ].to_numpy()

        if len(period_data) > 0:
            positions.append(position_mapping[i])
            data_list.append(period_data)
            colors_list.append(period_colors[period])

    if len(data_list) == 0:
        ax.text(0.5, 0.5, "No data", ha="center", va="center",
                fontsize=11, color="gray", transform=ax.transAxes)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        continue

    parts = ax.violinplot(
        data_list,
        positions=positions,
        widths=0.28,
        showmeans=False,
        showmedians=False,
        showextrema=False
    )

    for pc, pos, color in zip(parts["bodies"], positions, colors_list):
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
        pc.set_edgecolor(color)
        pc.set_linewidth(1.0)

        vertices = pc.get_paths()[0].vertices
        vertices[:, 0] = np.clip(vertices[:, 0], pos, pos + 0.28)

    for i, (pos, values) in enumerate(zip(positions, data_list)):
        q1, median, q3 = np.percentile(values, [25, 50, 75])

        ax.add_patch(
            Rectangle(
                (pos - 0.05, q1),
                0.10,
                q3 - q1,
                facecolor=colors_list[i],
                edgecolor="white",
                alpha=0.9,
                linewidth=1.0,
                zorder=3
            )
        )

        ax.plot(
            [pos - 0.10, pos + 0.10],
            [median, median],
            color="white",
            linewidth=1.5,
            zorder=4
        )

    for i, (pos, values) in enumerate(zip(positions, data_list)):
        jitter = np.random.uniform(-0.15, -0.02, len(values))

        ax.scatter(
            pos + jitter,
            values,
            alpha=0.6,
            s=2.5,
            color=colors_list[i],
            edgecolors="white",
            linewidths=0.3,
            zorder=2
        )

    panel_y_limit = country_data["Panel y-axis upper limit (km)"].iloc[0]

    ax.set_xlim(-0.25, 1.65)
    ax.set_ylim(0, panel_y_limit)
    ax.set_xticks([0, 0.45, 0.9, 1.35])
    ax.set_xticklabels(["I", "II", "III", "IV"], fontsize=13, ha="center")
    ax.set_yticks([0, panel_y_limit / 2, panel_y_limit])
    ax.set_yticklabels(
        [f"{int(value)}" for value in [0, panel_y_limit / 2, panel_y_limit]],
        fontsize=11
    )

    ax.set_title(country, fontsize=13, pad=6)
    ax.grid(axis="y", linestyle=":", alpha=0.3, linewidth=0.5)
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)

    if col == 0:
        ax.set_ylabel("Distance (km)", fontsize=12, labelpad=2)

legend_elements = [
    Rectangle((0, 0), 1, 1, fc=period_colors["Phase I: <2006"], alpha=0.7,
              edgecolor="gray", linewidth=0.5, label="Phase I: <2006"),
    Rectangle((0, 0), 1, 1, fc=period_colors["Phase II: 2006-2015"], alpha=0.7,
              edgecolor="gray", linewidth=0.5, label="Phase II: 2006-2015"),
    Rectangle((0, 0), 1, 1, fc=period_colors["Phase III: 2016-2019"], alpha=0.7,
              edgecolor="gray", linewidth=0.5, label="Phase III: 2016-2019"),
    Rectangle((0, 0), 1, 1, fc=period_colors["Phase IV: 2020-2024"], alpha=0.7,
              edgecolor="gray", linewidth=0.5, label="Phase IV: 2020-2024")
]

fig.legend(
    handles=legend_elements,
    loc="lower center",
    bbox_to_anchor=(0.43, -0.01),
    ncol=4,
    frameon=False,
    fontsize=13
)

figure_path = os.path.join(FIGURES, "fig2a.pdf")

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

source_data_fig2a = df_plot[
    [
        "country",
        "ai_year",
        "year_group",
        "nearest_distance_km",
        "Country pooled 99th percentile (km)",
        "Panel y-axis upper limit (km)"
    ]
].copy()

source_data_fig2a["Number of data centers in displayed country-period data"] = (
    source_data_fig2a.groupby(["country", "year_group"], observed=True)["nearest_distance_km"]
    .transform("size")
)

source_data_fig2a = source_data_fig2a.sort_values(
    ["country", "year_group", "ai_year"]
).rename(
    columns={
        "country": "Country",
        "ai_year": "Data center commissioning year",
        "year_group": "Commissioning period",
        "nearest_distance_km": "Distance to nearest operating power plant (km)"
    }
)

source_path = os.path.join(FIGURES, "source_data_fig2a.csv")
source_data_fig2a.to_csv(source_path, index=False)

print(f"Figure saved to: {figure_path}")
print(f"Source data saved to: {source_path}")

In [ ]:
source_data_fig2a = df_plot[
    [
        "country",
        "ai_year",
        "year_group",
        "nearest_distance_km",
        "Country pooled 99th percentile (km)",
        "Panel y-axis upper limit (km)"
    ]
].copy()

source_data_fig2a["Number of data centers in displayed country-period data"] = (
    source_data_fig2a
    .groupby(["country", "year_group"], observed=True)["nearest_distance_km"]
    .transform("size")
)

source_data_fig2a = source_data_fig2a.sort_values(
    ["country", "year_group", "ai_year"]
).rename(
    columns={
        "country": "Country",
        "ai_year": "Data center commissioning year",
        "year_group": "Commissioning period",
        "nearest_distance_km": "Distance to nearest operating power plant (km)"
    }
)

source_path = os.path.join(
    FIGURES,
    "source_data_fig2a.csv"
)

source_data_fig2a.to_csv(source_path, index=False)

print(f"Source data saved to: {source_path}")

In [ ]:
print(f"Total data-center observations shown in Fig. 2a: {len(source_data_fig2a):,}")

print("\nObservations shown by country:")
print(
    source_data_fig2a
    .groupby("Country", observed=True)
    .size()
    .rename("Number of data centers")
)

print("\nObservations shown by country and commissioning period:")
print(
    source_data_fig2a
    .groupby(["Country", "Commissioning period"], observed=True)
    .size()
    .unstack(fill_value=0)
)

In [ ]:
# b, calculation section: use the same displayed data-center sample as Fig. 2a

import os
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

# ==================== Settings ====================
TOP10_COUNTRIES = ["USA", "CHN", "JPN", "GBR", "DEU",
                   "RUS", "FRA", "CAN", "NLD", "AUS"]

# ==================== Load data ====================
oilgas_df = pd.read_excel(
    os.path.join(RAW, "Global-Oil-and-Gas-Plant-Tracker-GOGPT-January-2025.xlsx"),
    sheet_name="all"
)
oilgas_country = pd.read_excel(
    os.path.join(RAW, "Global-Oil-and-Gas-Plant-Tracker-GOGPT-January-2025.xlsx"),
    sheet_name="country"
)

coal_df = pd.read_excel(
    os.path.join(RAW, "Global-Coal-Plant-Tracker-January-2025.xlsx"),
    sheet_name="Units"
)
coal_country = pd.read_excel(
    os.path.join(RAW, "Global-Coal-Plant-Tracker-January-2025.xlsx"),
    sheet_name="country"
)

solar_df = pd.read_excel(
    os.path.join(RAW, "Global-Solar-Power-Tracker-February-2025.xlsx"),
    sheet_name="all"
)
solar_country = pd.read_excel(
    os.path.join(RAW, "Global-Solar-Power-Tracker-February-2025.xlsx"),
    sheet_name="country"
)

wind_df = pd.read_excel(
    os.path.join(RAW, "Global-Wind-Power-Tracker-February-2025.xlsx"),
    sheet_name="Data"
)
wind_country = pd.read_excel(
    os.path.join(RAW, "Global-Wind-Power-Tracker-February-2025.xlsx"),
    sheet_name="country"
)

ai_df = pd.read_excel(
    os.path.join(RAW, "SPGlobal_Export.xlsx"),
    sheet_name="Sheet1"
)
ai_country = pd.read_excel(
    os.path.join(RAW, "SPGlobal_Export.xlsx"),
    sheet_name="country"
)

# ==================== Process plant data ====================
def process_plant_data(plant_df, country_df, plant_type):
    plant_clean = plant_df.loc[
        plant_df["Status"].isin(["operating", "retired"])
    ].copy()

    for col in ["Latitude", "Longitude", "Start year", "Retired year", "Capacity (MW)"]:
        plant_clean[col] = pd.to_numeric(plant_clean[col], errors="coerce")

    plant_clean = plant_clean.dropna(subset=["Latitude", "Longitude", "Start year"])

    plant_clean = plant_clean.merge(
        country_df[["Country/Area", "Alpha-3 code"]],
        on="Country/Area",
        how="left"
    )

    plant_clean["plant_type"] = plant_type
    return plant_clean

all_plants = pd.concat(
    [
        process_plant_data(oilgas_df, oilgas_country, "Oil & Gas"),
        process_plant_data(coal_df, coal_country, "Coal"),
        process_plant_data(solar_df, solar_country, "Solar"),
        process_plant_data(wind_df, wind_country, "Wind"),
    ],
    ignore_index=True
)

all_plants = all_plants.loc[
    all_plants["Alpha-3 code"].isin(TOP10_COUNTRIES)
].copy()

# ==================== Process data-center data ====================
ai_clean = ai_df.copy()

for col in ["LATITUDE", "LONGITUDE", "YR_BUILT"]:
    ai_clean[col] = pd.to_numeric(ai_clean[col], errors="coerce")

ai_clean = ai_clean.dropna(subset=["LATITUDE", "LONGITUDE", "YR_BUILT"])

ai_clean = ai_clean.merge(
    ai_country[["COUNTRY", "Alpha-3 code"]],
    on="COUNTRY",
    how="left"
)

# China denotes mainland China only: do not merge HKG, MAC or TWN into CHN.
ai_clean = ai_clean.loc[
    ai_clean["Alpha-3 code"].isin(TOP10_COUNTRIES)
    & ai_clean["YR_BUILT"].between(1900, 2024)
].copy()

ai_clean["ai_year"] = ai_clean["YR_BUILT"].astype(int)

# ==================== Helpers ====================
def to_unit_sphere(latitude, longitude):
    lat = np.deg2rad(np.asarray(latitude, dtype=float))
    lon = np.deg2rad(np.asarray(longitude, dtype=float))

    return np.column_stack(
        [
            np.cos(lat) * np.cos(lon),
            np.cos(lat) * np.sin(lon),
            np.sin(lat),
        ]
    )

def classify_year(year):
    if year < 2006:
        return "<2006"
    if year <= 2015:
        return "2006-2015"
    if year <= 2019:
        return "2016-2019"
    return "2020-2024"

# ==================== Match each data center to its nearest eligible plant ====================
results = []

for country in TOP10_COUNTRIES:
    dc_country = ai_clean.loc[
        ai_clean["Alpha-3 code"] == country
    ].copy()

    plant_country = all_plants.loc[
        all_plants["Alpha-3 code"] == country
    ].copy()

    if dc_country.empty or plant_country.empty:
        continue

    for year, dc_year in dc_country.groupby("ai_year", sort=False):
        eligible_plants = plant_country.loc[
            (plant_country["Start year"] <= year)
            & (
                (plant_country["Status"] == "operating")
                | (
                    (plant_country["Status"] == "retired")
                    & (plant_country["Retired year"] >= year)
                )
            )
        ].copy()

        if eligible_plants.empty:
            continue

        plant_xyz = to_unit_sphere(
            eligible_plants["Latitude"],
            eligible_plants["Longitude"]
        )
        tree = cKDTree(plant_xyz)

        dc_xyz = to_unit_sphere(
            dc_year["LATITUDE"],
            dc_year["LONGITUDE"]
        )

        chord_distance, nearest_index = tree.query(dc_xyz, k=1)

        # Convert unit-sphere chord distance to great-circle distance in km.
        distance_km = 2 * 6371.0 * np.arcsin(
            np.clip(chord_distance / 2, 0, 1)
        )

        nearest_plants = eligible_plants.iloc[nearest_index].reset_index(drop=True)
        dc_year = dc_year.reset_index(drop=True)

        matched = pd.DataFrame(
            {
                "country": country,
                "ai_year": dc_year["ai_year"],
                "nearest_distance_km": distance_km,
                "nearest_plant_type": nearest_plants["plant_type"],
                "nearest_plant_capacity_mw": nearest_plants["Capacity (MW)"].fillna(0),
            }
        )

        results.append(matched)

results_df = pd.concat(results, ignore_index=True)

# ==================== Apply the Fig. 2a country-specific pooled 99th-percentile rule ====================
results_df["display_limit_km"] = (
    results_df.groupby("country", observed=True)["nearest_distance_km"]
    .transform(lambda x: x.quantile(0.99))
)

results_df = results_df.loc[
    results_df["nearest_distance_km"] <= results_df["display_limit_km"]
].copy()

results_df["year_group"] = results_df["ai_year"].map(classify_year)

# ==================== Summarize nearest-plant capacity composition ====================
results_df["coal_capacity_mw"] = np.where(
    results_df["nearest_plant_type"].eq("Coal"),
    results_df["nearest_plant_capacity_mw"],
    0
)
results_df["oilgas_capacity_mw"] = np.where(
    results_df["nearest_plant_type"].eq("Oil & Gas"),
    results_df["nearest_plant_capacity_mw"],
    0
)
results_df["solar_wind_capacity_mw"] = np.where(
    results_df["nearest_plant_type"].isin(["Solar", "Wind"]),
    results_df["nearest_plant_capacity_mw"],
    0
)

country_stats = (
    results_df.groupby(["country", "year_group"], observed=True)
    .agg(
        coal_cap_avg=("coal_capacity_mw", "sum"),
        oilgas_cap_avg=("oilgas_capacity_mw", "sum"),
        solar_wind_cap_avg=("solar_wind_capacity_mw", "sum"),
        total=("ai_year", "size"),
    )
    .reset_index()
)

plot_data = country_stats.copy()

plot_data["total_capacity"] = (
    plot_data["coal_cap_avg"]
    + plot_data["oilgas_cap_avg"]
    + plot_data["solar_wind_cap_avg"]
)

plot_data["coal_pct"] = (
    100 * plot_data["coal_cap_avg"] / plot_data["total_capacity"]
)
plot_data["oilgas_pct"] = (
    100 * plot_data["oilgas_cap_avg"] / plot_data["total_capacity"]
)
plot_data["solar_wind_pct"] = (
    100 * plot_data["solar_wind_cap_avg"] / plot_data["total_capacity"]
)

plot_data = plot_data.replace([np.inf, -np.inf], np.nan).fillna(0)

# Keep the existing output filename for downstream plotting code.
plot_data.to_csv(
    os.path.join(TEMP, "capacity_composition_data.csv"),
    index=False
)

print(
    "Fig. 2b data-center observations after the same country-specific "
    f"pooled 99th-percentile rule as Fig. 2a: {len(results_df):,}"
)
print(f"Saved: {os.path.join(TEMP, 'capacity_composition_data.csv')}")

In [ ]:
print("Top-10 mainland-China sample:", len(ai_clean))
print("Matched eligible-plant sample before p99:", len(results_df) + 68)
print("Final Fig. 2b / Fig. 2a display sample:", len(results_df))
print("Sum of country-period counts:", int(plot_data["total"].sum()))

assert len(results_df) == 6235
assert int(plot_data["total"].sum()) == 6235

In [ ]:
# b, plot section
import os
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
from matplotlib.patches import Rectangle

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"

plt.rcParams["hatch.linewidth"] = 0.5

# Read data
plot_data = pd.read_csv(
    os.path.join(TEMP, "capacity_composition_data.csv")
)

# ==================== Top 10 ====================
top10_countries = [
    "USA", "CHN", "JPN", "GBR", "DEU",
    "RUS", "FRA", "CAN", "NLD", "AUS"
]

plot_data["country"] = pd.Categorical(
    plot_data["country"],
    categories=top10_countries,
    ordered=True
)

plot_data["year_group"] = pd.Categorical(
    plot_data["year_group"],
    categories=["<2006", "2006-2015", "2016-2019", "2020-2024"],
    ordered=True
)

plot_data = plot_data.sort_values(["country", "year_group"])

# ==================== Colour scheme ====================
coal_color = "#000000"
oilgas_color = "#4F78B5"
solar_wind_color = "#C77DA8"

# Hatch patterns for commissioning periods
hatch_patterns = ["", "......", "//////", "xxxxxx"]
period_labels = ["<2006", "2006-2015", "2016-2019", "2020-2024"]

# ==================== Create figure ====================
fig, ax = plt.subplots(figsize=(6, 2.2), dpi=300)

x_positions = []
x_labels = []

bar_width = 0.12
gap_between_bars = 0.03
gap_between_countries = 0.2

current_x = 0

for country in top10_countries:
    country_data = plot_data[plot_data["country"] == country]

    for j, period in enumerate(period_labels):
        x_pos = current_x + j * (bar_width + gap_between_bars)
        x_positions.append(x_pos)

        period_data = country_data[country_data["year_group"] == period]

        if len(period_data) > 0:
            coal_pct = period_data["coal_pct"].values[0]
            oilgas_pct = period_data["oilgas_pct"].values[0]
            solar_wind_pct = period_data["solar_wind_pct"].values[0]

            hatch = hatch_patterns[j]

            # Rasterize only hatched bars. This prevents Illustrator from
            # remapping PDF hatch patterns when the figure is copied.
            rasterize_hatch = hatch != ""

            ax.bar(
                x_pos,
                coal_pct,
                bar_width,
                color=coal_color,
                edgecolor="white",
                linewidth=0.2,
                hatch=hatch,
                rasterized=rasterize_hatch
            )

            ax.bar(
                x_pos,
                oilgas_pct,
                bar_width,
                bottom=coal_pct,
                color=oilgas_color,
                edgecolor="white",
                linewidth=0.2,
                hatch=hatch,
                rasterized=rasterize_hatch
            )

            ax.bar(
                x_pos,
                solar_wind_pct,
                bar_width,
                bottom=coal_pct + oilgas_pct,
                color=solar_wind_color,
                edgecolor="white",
                linewidth=0.2,
                hatch=hatch,
                rasterized=rasterize_hatch
            )

    country_center = current_x + (3 * (bar_width + gap_between_bars)) / 2
    x_labels.append(country_center)

    current_x += 4 * (bar_width + gap_between_bars) + gap_between_countries

# ==================== Axes ====================
ax.set_xlim(-0.3, current_x - gap_between_countries + 0.3)
ax.set_ylim(0, 100)

ax.set_xticks(x_labels)
ax.set_xticklabels(top10_countries, fontsize=6, family="Arial")

ax.set_yticks([0, 25, 50, 75, 100])
ax.set_yticklabels(["0", "25", "50", "75", "100"], fontsize=6, family="Arial")
ax.set_ylabel("Percentage of Total Capacity (%)", fontsize=6, family="Arial")

ax.grid(
    axis="y",
    linestyle="-",
    alpha=0.2,
    linewidth=0.5,
    zorder=0,
    color="gray"
)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(1)
ax.spines["bottom"].set_linewidth(1)

# ==================== Legends ====================
legend_x = 0.99
legend_y_start = 0.75
legend_spacing = 0.08

color_legend_elements = [
    {"color": coal_color, "label": "Coal"},
    {"color": oilgas_color, "label": "Oil & Gas"},
    {"color": solar_wind_color, "label": "Solar & Wind"}
]

for i, elem in enumerate(color_legend_elements):
    y_pos = legend_y_start - i * legend_spacing

    rect = Rectangle(
        (legend_x, y_pos),
        0.02,
        0.03,
        transform=ax.transAxes,
        facecolor=elem["color"],
        edgecolor="black",
        linewidth=0.5,
        clip_on=False
    )
    ax.add_patch(rect)

    ax.text(
        legend_x + 0.03,
        y_pos + 0.015,
        elem["label"],
        transform=ax.transAxes,
        fontsize=6,
        family="Arial",
        va="center",
        ha="left"
    )

hatch_legend_elements = [
    {"hatch": "", "label": "<2006"},
    {"hatch": "......", "label": "2006-2015"},
    {"hatch": "//////", "label": "2016-2019"},
    {"hatch": "xxxxxx", "label": "2020-2024"}
]

legend_y_start2 = legend_y_start - 4 * legend_spacing

for i, elem in enumerate(hatch_legend_elements):
    y_pos = legend_y_start2 - i * legend_spacing

    rect = Rectangle(
        (legend_x, y_pos),
        0.02,
        0.03,
        transform=ax.transAxes,
        facecolor="white",
        edgecolor="black",
        linewidth=0.2,
        hatch=elem["hatch"],
        rasterized=elem["hatch"] != "",
        clip_on=False
    )
    ax.add_patch(rect)

    ax.text(
        legend_x + 0.03,
        y_pos + 0.015,
        elem["label"],
        transform=ax.transAxes,
        fontsize=6,
        family="Arial",
        va="center",
        ha="left"
    )

plt.tight_layout()

plt.savefig(
    os.path.join(FIGURES, "fig2b.pdf"),
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="none"
)

plt.show()

print(f"\nFigure saved to: {os.path.join(FIGURES, 'fig2b.pdf')}")
print(f"Countries displayed: {', '.join(top10_countries)}")

In [ ]:
# ==================== Source data: Fig. 2b ====================
source_data_fig2b = (
    plot_data[
        [
            "country",
            "year_group",
            "total",
            "coal_cap_avg",
            "oilgas_cap_avg",
            "solar_wind_cap_avg",
            "total_capacity",
            "coal_pct",
            "oilgas_pct",
            "solar_wind_pct"
        ]
    ]
    .sort_values(["country", "year_group"])
    .rename(
        columns={
            "country": "Country",
            "year_group": "Commissioning period",
            "total": "Number of data centers",
            "coal_cap_avg": "Capacity of nearest coal plants (MW)",
            "oilgas_cap_avg": "Capacity of nearest oil and gas plants (MW)",
            "solar_wind_cap_avg": "Capacity of nearest solar and wind plants (MW)",
            "total_capacity": "Total capacity of nearest plants (MW)",
            "coal_pct": "Coal capacity share (%)",
            "oilgas_pct": "Oil and gas capacity share (%)",
            "solar_wind_pct": "Solar and wind capacity share (%)"
        }
    )
)

source_data_fig2b.to_csv(
    os.path.join(FIGURES, "source_data_fig2b.csv"),
    index=False
)

print(f"Source data saved to: {os.path.join(FIGURES, 'source_data_fig2b.csv')}")

In [ ]:
#sum crypto data center

In [ ]:
from pathlib import Path
import pandas as pd
import os

# Path setting
BASE_PATH = Path.cwd().parent.parent
RAW = BASE_PATH / 'Data' / 'raw'

# Read data
df = pd.read_excel(os.path.join(RAW, 'SPGlobal_Export.xlsx'), sheet_name='Sheet1')

# Use the same valid-sample filter as Fig. 1b
df = df[
    df['YR_BUILT'].notna() &
    df['LATITUDE'].notna() &
    df['LONGITUDE'].notna() &
    (df['YR_BUILT'] != 2025)
].copy()

crypto_type = 'Crypto Mining Data Center'

total_n = len(df)
crypto_n = (df['SECONDARY_PPTY_TYPE'] == crypto_type).sum()
crypto_share = crypto_n / total_n * 100

print("Crypto mining data center summary")
print("-" * 40)
print(f"Total valid data centers: {total_n:,}")
print(f"Crypto mining data centers: {crypto_n:,}")
print(f"Share of crypto mining data centers: {crypto_share:.2f}%")

In [38]:
import sys
print(sys.version)

3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:54:21) [Clang 16.0.6 ]
